In [16]:
keywords = [
    # Macro economie
    "inflation", "deflation", "interest rates", "rate hike", "rate cut",
    "recession", "economic slowdown", "gdp", "consumer spending",
    "unemployment", "job growth", "employment", "wage growth",
    "housing market", "real estate", "credit", "debt", "liquidity",

    # Centrale banken / beleid
    "federal reserve", "fed", "ecb", "central bank",
    "monetary policy", "quantitative easing", "quantitative tightening",
    "bond yields", "treasury yields",

    # Markt / trading termen
    "stock market", "stocks", "equities", "bull market", "bear market",
    "market rally", "market crash", "selloff", "volatility", "correction",
    "overvalued", "undervalued", "bubble",

    # Bedrijven / earnings
    "earnings", "earnings report", "revenue", "profit", "guidance",
    "forecast", "downgrade", "upgrade", "ipo", "merger", "acquisition",

    # Tech / AI (super belangrijk momenteel)
    "ai", "artificial intelligence", "machine learning", "automation",
    "semiconductors", "chips", "nvidia", "openai", "cloud computing",

    # Grote namen (markt movers)
    "elon musk", "tesla", "apple", "microsoft", "amazon", "google", "meta",

    # Politiek / geopolitiek
    "trump", "biden", "white house", "election",
    "war", "conflict", "sanctions", "china", "russia", "ukraine",
    "middle east", "trade war", "tariffs",

    # Grondstoffen / alternatieven
    "oil", "gold", "commodities", "energy prices", "gas prices",

    # Sentiment / angst
    "fear", "panic", "uncertainty", "risk", "risk-off", "risk-on",
    "investor sentiment",

    # Crypto (vaak leading indicator voor risk appetite)
    "bitcoin", "crypto", "cryptocurrency", "blockchain",

    # Banken / financiële stress
    "banking crisis", "bank failure", "liquidity crisis",
    "credit crunch"
]

sections = ["Business Day", "Health", "Education", "Science", "Blogs", "U.S.", "New York", "Real Estate", "Washington", "World", "Your Money", "Technology", "Job Market"]

In [23]:
from langchain_groq import ChatGroq
import os
import json

from dotenv import load_dotenv
load_dotenv()

True

In [27]:
GROQ_API_KEY = os.getenv("groq_API")

LLM = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)


def prompt(headline,abstract,section):

  PROMPT = f"""You are a financial market analyst specialized in evaluating news impact on stock markets.

    INPUT:
    1. Headline: {headline}
    2. Abstract: {abstract}
    3. Section: {section}
    
    Base your score of of this info, rules and extra's you find below:

  For every article you receive, return ONLY a valid JSON object with exactly two fields:
  - "positivity_score": float between -1.0 and 1.0
    -1.0 = strongly negative market sentiment
    0.0 = neutral
    1.0 = strongly positive market sentiment

  - "influence_score": float between 0.0 and 1.0
    0.0 = no market influence expected
    1.0 = extreme market-moving potential

  Scoring guidelines for influence_score:
  - Central bank decisions, rate changes, major earnings beats/misses → 0.8–1.0
  - Geopolitical events, regulatory changes, major M&A → 0.6–0.8
  - Economic indicators (CPI, jobs, GDP) → 0.5–0.7
  - Analyst upgrades/downgrades, sector news → 0.3–0.5
  - General business news, minor updates → 0.1–0.3
  - Fluff, opinion pieces, low-relevance articles → 0.0–0.1

  Return ONLY the JSON object. No explanation, no markdown, no extra text."""
  return json.loads(LLM.invoke(PROMPT).content.strip())

In [31]:
import requests
import time
import os
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import re

load_dotenv()

API_KEY = os.getenv("NYT_API_KEY")
START_YEAR = 2026
CURRENT_YEAR = datetime.now().year
CURRENT_MONTH = datetime.now().month

def fetch_and_save_nyt_data():
    for year in range(CURRENT_YEAR, CURRENT_YEAR + 1):
        year_data = []
        
        # Bepaal tot welke maand we moeten gaan voor het huidige jaar
        end_month = CURRENT_MONTH if year == CURRENT_YEAR else 12
        
        for month in range(1, end_month + 1):
            print(f"Ophalen: {year}-{month}...")
            
            url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json?api-key={API_KEY}"
            
            try:
                response = requests.get(url)
                
                if response.status_code == 200:
                    data = response.json()
                    articles = data['response']['docs']
                    
                    # Alleen relevante velden selecteren om CSV compact te houden
                    # Definieer je keywords
                    keyword_patterns = [
                                re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
                                for word in keywords
                            ]
                    for art in articles:
                        # Check of keywords voorkomen in de kop of de samenvatting
                        headline = art.get('headline', {}).get('main', "").lower()
                        abstract = art.get('abstract', "").lower()
                        section = art.get('section_name')
                        # Filter: check of één van de keywords in de tekst staat
                        if any(pattern.search(headline) for pattern in keyword_patterns) and section in sections:
                            response_llm = prompt(headline, abstract, section)
                            print(response_llm)
                            year_data.append({
                                'pub_date': art.get('pub_date'),
                                'headline': headline,
                                'abstract': abstract,
                                'positivityScore' : response_llm['positivity_score'],
                                'influenceScore' : response_llm['influence_score'],
                                'section': art.get('section_name'),
                                'web_url': art.get('web_url')
                            })                
                elif response.status_code == 429:
                    print("  -> Rate limit bereikt! 60 seconden pauze...")
                    time.sleep(60)
                    # Je zou hier een 'retry' kunnen inbouwen
                else:
                    print(f"  -> Fout {response.status_code} bij {year}-{month}")

            except Exception as e:
                print(f"  -> Er ging iets mis: {e}")

            # Cruciaal: wacht 10 seconden tussen elke maand om 429 errors te voorkomen
            time.sleep(10)

        # Sla data per jaar op als een CSV
        if year_data:
            df = pd.DataFrame(year_data)
            # Verwijder duplicates op basis van headline
            df = df.drop_duplicates(subset=['headline'])
            filename = f"nyt_data_{year}.csv"
            df.to_csv(filename, index=False, encoding='utf-8')
            print(f"Jaar {year} succesvol opgeslagen in {filename} met {len(df)} articles✔️")

if __name__ == "__main__":
    if not API_KEY:
        print("Fout: Geen API_KEY gevonden in .env bestand.")
    else:
        fetch_and_save_nyt_data()

Ophalen: 2026-1...
{'positivity_score': -0.2, 'influence_score': 0.6}
{'positivity_score': 0.0, 'influence_score': 0.4}
{'positivity_score': -0.5, 'influence_score': 0.7}
{'positivity_score': 0.0, 'influence_score': 0.6}
{'positivity_score': -0.8, 'influence_score': 0.7}
{'positivity_score': -0.2, 'influence_score': 0.4}
{'positivity_score': 0.0, 'influence_score': 0.0}
{'positivity_score': -0.2, 'influence_score': 0.6}
  -> Er ging iets mis: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kh454fb0ftes0vym1e04mnrn` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 422. Please try again in 6m4.608s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


KeyboardInterrupt: 

In [32]:
# %%
sections = ["Business Day","Health","Education","Science","Blogs","U.S.","New York",
            "Real Estate","Washington","World","Your Money","Technology","Job Market"]

# %%
import requests
import time
import os
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

load_dotenv()

API_KEY = os.getenv("NYT_API_KEY")
CURRENT_YEAR = datetime.now().year
CURRENT_MONTH = datetime.now().month

# %%
# FinBERT laden
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

labels = ["negative", "neutral", "positive"]

# %%
def get_scores(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = F.softmax(outputs.logits, dim=1).numpy()[0]
    
    prob_dict = dict(zip(labels, probs))
    
    # Positivity score (-1 tot 1)
    positivity = prob_dict["positive"] - prob_dict["negative"]
    
    # Influence = hoe zeker model is
    influence = max(probs)
    
    return float(positivity), float(influence)

# %%
def fetch_and_save_nyt_data():
    for year in range(CURRENT_YEAR, CURRENT_YEAR + 1):
        year_data = []
        
        end_month = CURRENT_MONTH if year == CURRENT_YEAR else 12
        
        for month in range(1, end_month + 1):
            print(f"Ophalen: {year}-{month}...")
            
            url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json?api-key={API_KEY}"
            
            try:
                response = requests.get(url)
                
                if response.status_code == 200:
                    data = response.json()
                    articles = data['response']['docs']

                    for art in articles:
                        headline = art.get('headline', {}).get('main', "")
                        abstract = art.get('abstract', "")
                        section = art.get('section_name')

                        if section in sections:
                            text = headline + " " + abstract
                            
                            positivity, influence = get_scores(text)

                            print({
                                "positivity": positivity,
                                "influence": influence
                            })

                            year_data.append({
                                'pub_date': art.get('pub_date'),
                                'headline': headline,
                                'abstract': abstract,
                                'positivityScore': positivity,
                                'influenceScore': influence,
                                'section': section,
                                'web_url': art.get('web_url')
                            })

                elif response.status_code == 429:
                    print("  -> Rate limit bereikt! 60 seconden pauze...")
                    time.sleep(60)

                else:
                    print(f"  -> Fout {response.status_code} bij {year}-{month}")

            except Exception as e:
                print(f"  -> Er ging iets mis: {e}")

            time.sleep(10)

        if year_data:
            df = pd.DataFrame(year_data)
            df = df.drop_duplicates(subset=['headline'])
            filename = f"nyt_data_{year}.csv"
            df.to_csv(filename, index=False, encoding='utf-8')
            print(f"Jaar {year} succesvol opgeslagen in {filename} met {len(df)} articles ✔️")

# %%
if __name__ == "__main__":
    if not API_KEY:
        print("Fout: Geen API_KEY gevonden in .env bestand.")
    else:
        fetch_and_save_nyt_data()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 23153.22it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ophalen: 2026-1...
{'positivity': -0.46209004521369934, 'influence': 0.7126052975654602}
{'positivity': 0.5483561158180237, 'influence': 0.6160504817962646}
{'positivity': 0.22965431213378906, 'influence': 0.7187556028366089}
{'positivity': -0.21485929191112518, 'influence': 0.5864708423614502}
{'positivity': 0.04535963386297226, 'influence': 0.8943071961402893}
{'positivity': 0.09137358516454697, 'influence': 0.8702462315559387}
{'positivity': -0.11908519268035889, 'influence': 0.5867089033126831}
{'positivity': 0.1083931028842926, 'influence': 0.37249690294265747}
{'positivity': 0.815380871295929, 'influence': 0.897996187210083}
{'positivity': 0.6039137840270996, 'influence': 0.795447587966919}
{'positivity': 0.8924307823181152, 'influence': 0.9200279712677002}
{'positivity': 0.3028368055820465, 'influence': 0.5945447683334351}
{'positivity': 0.7052910327911377, 'influence': 0.8439145684242249}
{'positivity': 0.10938270390033722, 'influence': 0.8522757887840271}
{'positivity': 0.2024

KeyboardInterrupt: 